# Lab 5 · Explain a decade of returns

**Week 5 · Performance attribution** · ≈ 75 minutes

Take the momentum strategy (lookback 6, top 5, hold 3) from 2011 onwards and explain its returns against a
market-cap benchmark of the same 25 portfolios: first with Brinson–Fachler by **size segment**, then with
a factor regression on the Asia-Pacific five factors plus momentum.

**Data.** Monthly returns from the Kenneth R. French Data Library (Tuck School of Business, Dartmouth), *Asia Pacific ex Japan* region: Australia, **Hong Kong**, New Zealand and Singapore, July 1990 onwards. The setup cell downloads it once and caches it in `finex_data/`.

**How this lab works.** Run the setup cell, then work top to bottom. Each exercise has a `# TODO` cell followed by an automatic check (✓ / ✗ with a hint). When every check passes, the last cell prints a **completion code**: paste it into the course page to complete the lab.

In [1]:
# FinEx Learning · lab setup. Run this cell first; it only defines helpers.
# Data: Kenneth R. French Data Library (Tuck School of Business, Dartmouth),
# Asia Pacific ex Japan series (Australia, Hong Kong, New Zealand, Singapore),
# downloaded on first use and cached next to the notebook.
import hashlib
import io
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

FRENCH = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
CACHE = Path("finex_data")
CACHE.mkdir(exist_ok=True)


def _lines(name):
    path = CACHE / f"{name}.csv"
    if not path.exists():
        with urllib.request.urlopen(f"{FRENCH}{name}_CSV.zip", timeout=60) as response:
            archive = zipfile.ZipFile(io.BytesIO(response.read()))
            path.write_bytes(archive.read(archive.namelist()[0]))
    return path.read_text(encoding="latin-1").splitlines()


def load_french(name, table=None, percent=True):
    """One monthly table from a French data library file, indexed by month.

    `table` is the title above the table (e.g. "Average Value Weighted Returns -- Monthly");
    None means the first monthly table. Returns are converted to decimals.
    """
    tables, title, current = {}, "", None
    for line in _lines(name):
        cells = [c.strip() for c in line.split(",")]
        if line.lstrip().startswith(","):
            current = []
            tables.setdefault(title or "first", (cells[1:], current))
            continue
        if current is not None and len(cells[0]) == 6 and cells[0].isdigit():
            current.append(cells)
            continue
        if line.strip():
            title, current = line.strip(), None
    key = table or next(k for k, (_, rows) in tables.items() if rows)
    header, rows = tables[key]
    frame = pd.DataFrame(
        [r[1:] for r in rows],
        columns=header,
        index=pd.PeriodIndex([r[0] for r in rows], freq="M"),
    ).astype(float).replace(-99.99, np.nan)
    return frame / 100 if percent else frame


def check(name, condition, hint):
    """Automatic check: passes quietly, or explains what to look at."""
    if not bool(condition):
        raise AssertionError(f"✗ {name}. Hint: {hint}")
    print(f"✓ {name}")


def completion_code(lab):
    code = hashlib.sha256(f"finex-quant-{lab}".encode()).hexdigest()[:6].upper()
    print(f"\nAll checks passed. Your completion code for the course page: FINEX-{lab.upper()}-{code}")


def annualise(returns):
    """Mean, volatility, Sharpe and t-stat of monthly excess returns, annualised."""
    r = returns.dropna()
    mean, vol = r.mean() * 12, r.std() * np.sqrt(12)
    return pd.Series({"mean": mean, "vol": vol, "sharpe": mean / vol,
                      "t_stat": r.mean() / r.std() * np.sqrt(len(r)), "months": len(r)})


print("Setup complete. Data source: Kenneth R. French Data Library · Asia Pacific ex Japan.")


Setup complete. Data source: Kenneth R. French Data Library · Asia Pacific ex Japan.


In [2]:
import warnings
warnings.filterwarnings("ignore")
import statsmodels.api as sm

returns = load_french("Asia_Pacific_ex_Japan_25_Portfolios_ME_BE-ME", "Average Value Weighted Returns -- Monthly")
firms = load_french("Asia_Pacific_ex_Japan_25_Portfolios_ME_BE-ME", "Number of Firms in Portfolios", percent=False)
avg_size = load_french("Asia_Pacific_ex_Japan_25_Portfolios_ME_BE-ME", "Average Firm Size", percent=False)
five = load_french("Asia_Pacific_ex_Japan_5_Factors")
wml = load_french("Asia_Pacific_ex_Japan_MOM_Factor")["WML"]
segment = pd.Series([f"Size {i // 5 + 1}" for i in range(25)], index=returns.columns)

def momentum_strategy(returns, lookback, top_k, hold):
    """Long-only: equal-weight the top_k portfolios by past return, rebalanced every `hold` months."""
    score = (1 + returns).rolling(lookback - 1).apply(np.prod, raw=True).shift(1) - 1
    rows, weights, w_prev = [], [], np.zeros(returns.shape[1])
    for i in range(lookback + 1, len(returns)):
        if (i - lookback - 1) % hold == 0:
            w = np.zeros(returns.shape[1])
            w[returns.columns.get_indexer(score.iloc[i - 1].nlargest(top_k).index)] = 1 / top_k
        else:                                  # let weights drift between rebalances
            w = w_prev * (1 + returns.iloc[i - 1].values)
            w = w / w.sum()
        turnover = np.abs(w - w_prev).sum() / 2 if i > lookback + 1 else 1.0
        rows.append((returns.index[i], w @ returns.iloc[i].values, turnover))
        weights.append(w)
        w_prev = w
    frame = pd.DataFrame(rows, columns=["month", "ret", "turnover"]).set_index("month")
    return frame, pd.DataFrame(weights, index=frame.index, columns=returns.columns)

result, weights = momentum_strategy(returns, lookback=6, top_k=5, hold=3)
cap = (firms * avg_size).shift(1)
bench_weights = cap.div(cap.sum(axis=1), axis=0).loc[weights.index]
period = slice("2011-01", None)
w_p, w_b, r = weights.loc[period], bench_weights.loc[period], returns.loc[weights.index].loc[period]

## Exercise 1 · Brinson–Fachler by size segment, every month

For each month and segment *i*: segment weights $w_i$, segment returns $R_i$ (weight-averaged inside the
segment; use the benchmark return if the portfolio holds nothing there), total benchmark return $R^B$:

* Allocation $(w_i^P - w_i^B)(R_i^B - R^B)$
* Selection $w_i^B(R_i^P - R_i^B)$
* Interaction $(w_i^P - w_i^B)(R_i^P - R_i^B)$

In [3]:
def segment_view(w, rets):
    weight = w.T.groupby(segment).sum().T
    contrib = (w * rets).T.groupby(segment).sum().T
    return weight, contrib / weight.replace(0, np.nan)

wp_seg, rp_seg = segment_view(w_p, r)
wb_seg, rb_seg = segment_view(w_b, r)
rp_seg = rp_seg.fillna(rb_seg)
RB = (w_b * r).sum(axis=1)
effects = pd.DataFrame({
    "allocation": ((wp_seg - wb_seg) * rb_seg.sub(RB, axis=0)).sum(axis=1),
    "selection": (wb_seg * (rp_seg - rb_seg)).sum(axis=1),
    "interaction": ((wp_seg - wb_seg) * (rp_seg - rb_seg)).sum(axis=1),
    "active": (w_p * r).sum(axis=1) - RB,
})
(effects.mean() * 12).round(4)

allocation    -0.0011
selection      0.0054
interaction   -0.0007
active         0.0037
dtype: float64

In [4]:
check("Effects sum to the active return every month",
      isinstance(effects, pd.DataFrame) and np.allclose(effects[["allocation", "selection", "interaction"]].sum(axis=1), effects["active"], atol=1e-10),
      "fill segment returns where the portfolio has no weight with the benchmark segment return")

✓ Effects sum to the active return every month


## Exercise 2 · What factors were you paid for?

Regress the strategy's monthly excess return (from 2011) on `Mkt-RF`, `SMB`, `HML`, `RMW`, `CMA` and `WML`
with HAC (6 lags) errors. Report the exposures, the annualised alpha and R².

In [5]:
excess = (result["ret"] - five["RF"].reindex(result.index)).loc[period]
X = pd.concat([five[["Mkt-RF", "SMB", "HML", "RMW", "CMA"]], wml.rename("WML")], axis=1).loc[excess.index]
factor_model = sm.OLS(excess, sm.add_constant(X)).fit(cov_type="HAC", cov_kwds={"maxlags": 6})
print(f"Annualised alpha {factor_model.params['const'] * 12:.2%} (t = {factor_model.tvalues['const']:.2f}) · R² {factor_model.rsquared:.2f}")
factor_model.params.round(3)

Annualised alpha 0.12% (t = 0.09) · R² 0.92


const     0.000
Mkt-RF    1.006
SMB       0.325
HML       0.174
RMW      -0.055
CMA      -0.227
WML       0.067
dtype: float64

In [6]:
check("Model includes all six factors",
      factor_model is not None and set(["Mkt-RF", "SMB", "HML", "RMW", "CMA", "WML"]) <= set(factor_model.params.index),
      "concatenate the five factors with WML")
check("A momentum strategy loads positively on WML", factor_model is not None and factor_model.params["WML"] > 0,
      "the exposure to WML should be positive")
completion_code("lab5")

✓ Model includes all six factors
✓ A momentum strategy loads positively on WML

All checks passed. Your completion code for the course page: FINEX-LAB5-997117


## Reflect: write the one-page narrative

1. Where did the active return come from: allocation across size segments, or selection within them?
2. Which factor exposures were intended, and which were accidental?
3. Is there alpha left after the six factors? Would you charge active fees for this strategy?